# 2D Piecewise Algebraic Implicit Splines — Basic Polygon Demo

Demonstrates the **Li & Tian (2009)** implicit-spline construction for several
hard-coded polygons covering convex, concave, holes, and 2D partitions.

**Reference:**  
Li, Q. & Tian, J. (2009). *2D Piecewise Algebraic Splines for Implicit Modeling.*  
ACM Transactions on Graphics, 28(3). DOI: [10.1145/1516522.1516524](https://doi.org/10.1145/1516522.1516524)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QL-UoHull/Implicit-Spline/blob/main/notebooks/01_basic_polygon.ipynb)


## Setup

Install / import dependencies.

In [ ]:
# If running in Google Colab, clone the repo so imports work.
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Implicit-Spline'):
        os.system('git clone https://github.com/QL-UoHull/Implicit-Spline.git')
    sys.path.insert(0, 'Implicit-Spline/python')
else:
    # Robust local path setup: works when launched from repo root or notebooks/
    cwd = Path.cwd().resolve()
    candidates = [cwd / 'python', cwd.parent / 'python', cwd / 'Implicit-Spline' / 'python']
    for candidate in candidates:
        if candidate.is_dir():
            sys.path.insert(0, str(candidate))
            break
    else:
        raise RuntimeError('Could not locate the repository\'s python/ directory.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from implicit_spline import imp_spline_2d, draw_imp_spline
from implicit_spline.visualization import (
    draw_surface, compare_delta, compare_n, panel_delta_shapes,
    partition_basis_surfaces, draw_polygon_outline, make_grid
)
from implicit_spline.core import H

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120})
print('Imports OK')


## 1  The smooth step function H(t, δ, n)

The building block of the construction is a smooth Heaviside-like function
`H(t, delta, n)` that transitions from 0 to 1 over the interval `[0, delta]`
and is C^n continuous at both endpoints.

In [ ]:
t = np.linspace(-0.3, 1.3, 400)
delta = 1.0

fig, ax = plt.subplots(figsize=(7, 3.5))
for nn in [1, 2, 3]:
    ax.plot(t, H(t, delta, nn), label=f'n={nn}')
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.axvline(delta, color='k', lw=0.8, ls='--')
ax.axhline(0, color='k', lw=0.5)
ax.axhline(1, color='k', lw=0.5)
ax.set_xlabel('t'); ax.set_ylabel('H(t)')
ax.set_title(r'Smooth step function $H(t,\,\delta,\,n)$ for $\delta=1$')
ax.legend(); plt.tight_layout(); plt.show()


## 2  Unit square

Vertices are given in **counter-clockwise** order (the function auto-corrects
clockwise input).  The contour plot shows the implicit function value;
the white curve is the iso-level at `f = 0.5` and the dashed red line
is the original polygon.

In [ ]:
# Parameters
delta = 0.12   # transition bandwidth
n     = 2      # smoothness order
N     = 300    # grid resolution


In [ ]:
P_square = np.array([[0, 0], [1, 0], [1, 1], [0, 1]], dtype=float)

fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
draw_imp_spline(P_square, delta=delta, n=n, N=N, ax=ax1,
                title='Contour plot')
draw_surface(P_square, delta=delta, n=n, N=80, ax=ax2,
             title='Surface plot')
fig.suptitle('Unit square', fontsize=13)
plt.tight_layout(); plt.show()


## 3  Equilateral triangle

In [ ]:
P_tri = np.array([[0, 0], [2, 0], [1, np.sqrt(3)]], dtype=float)

fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
draw_imp_spline(P_tri, delta=delta, n=n, N=N, ax=ax1,
                title='Contour plot')
draw_surface(P_tri, delta=delta, n=n, N=80, ax=ax2,
             title='Surface plot')
fig.suptitle('Equilateral triangle', fontsize=13)
plt.tight_layout(); plt.show()


## 4  Regular pentagon

In [ ]:
theta  = np.linspace(np.pi/2, np.pi/2 + 2*np.pi, 6)[:-1]  # 5 vertices, start at top
P_pent = np.column_stack([np.cos(theta), np.sin(theta)])

fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
draw_imp_spline(P_pent, delta=delta, n=n, N=N, ax=ax1,
                title='Contour plot')
draw_surface(P_pent, delta=delta, n=n, N=80, ax=ax2,
             title='Surface plot')
fig.suptitle('Regular pentagon', fontsize=13)
plt.tight_layout(); plt.show()


## 5  Effect of the bandwidth parameter δ

Increasing δ widens the smooth transition zone near each edge and rounds
corners more aggressively.

In [ ]:
P_irreg = np.array([[0, 0], [2, 0], [2.5, 0.8], [1.5, 1.8], [-0.2, 1.2]])
compare_delta(P_irreg, deltas=(0.05, 0.15, 0.35), n=2, N=N)
plt.show()


## 6  Effect of the smoothness order n

Higher *n* gives smoother transitions near the boundary.

In [ ]:
P_sq = np.array([[0, 0], [1, 0], [1, 1], [0, 1]], dtype=float)
d = 0.15

compare_n(P_sq, delta=d, n_values=(1, 2, 3), N=N)
plt.show()


## 7  Concave / freeform polygon demos

The method can still be used for concave control polygons. This section makes the
concave case explicit and shows paper-style contour evolution with varying `\delta`.

In [ ]:
P_concave = np.array([
    [-1.00, -0.62], [-0.20, -0.20], [0.15, -0.76], [1.02, -0.25],
    [0.30, 0.02], [0.92, 0.82], [0.00, 0.40], [-0.88, 0.92], [-0.55, 0.02],
], dtype=float)

fig, ax = plt.subplots(figsize=(6.0, 5.0))
draw_imp_spline(P_concave, delta=0.18, n=2, N=N, ax=ax,
                title=r'Concave polygon ($\delta=0.18$, $n=2$)')
plt.tight_layout(); plt.show()

panel_delta_shapes(
    P_concave,
    deltas=(0.05, 0.10, 0.16, 0.24, 0.34, 0.46),
    n=2,
    N=300,
    layout=(2, 3),
    title='Concave polygon: contour evolution under increasing $\\delta$',
)
plt.show()

## 8  Polygon with holes

A simple way to model holes is to compose an outer polygon field with
complements of inner polygon fields.

In [ ]:
outer = np.array([[-1.10, -0.88], [-1.00, 0.90], [1.12, 0.92], [1.18, -0.90]], dtype=float)
hole1 = np.array([[-0.72, 0.30], [-0.44, -0.30], [0.10, 0.44]], dtype=float)
hole2 = np.array([[0.34, -0.26], [0.92, 0.16], [0.98, -0.52]], dtype=float)

all_pts = np.vstack([outer, hole1, hole2])
X, Y = make_grid(all_pts, N=300, pad_fraction=0.18)
Z = imp_spline_2d(X, Y, outer, delta=0.18, n=2)
for hole in (hole1, hole2):
    Z *= (1.0 - imp_spline_2d(X, Y, hole, delta=0.18, n=2))
Z = np.clip(Z, 0.0, 1.0)

fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
cf = ax1.contourf(X, Y, Z, levels=20, cmap='viridis')
plt.colorbar(cf, ax=ax1)
ax1.contour(X, Y, Z, levels=[0.5], colors='white', linewidths=2)
draw_polygon_outline(outer, ax=ax1, linestyle=':', color='0.55', linewidth=0.9, marker='o', markersize=3)
for hole in (hole1, hole2):
    draw_polygon_outline(hole, ax=ax1, linestyle=':', color='0.40', linewidth=0.9, marker='o', markersize=3)
ax1.set_title('Contour with holes')

ax2.plot_wireframe(X, Y, Z, rstride=5, cstride=5, color='0.35', linewidth=0.45)
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('f(x,y)')
ax2.set_title('Wireframe surface')
fig.suptitle('Polygon with holes', fontsize=13)
plt.tight_layout(); plt.show()

## 9  Multi-polygon 2D partition

This section explicitly shows a polygon partition net and the corresponding
family / sum of basis-function surfaces.

In [ ]:
partition_polygons = [
    np.array([[-0.95, -0.42], [-0.55,  0.10], [-0.55, -1.00]]),
    np.array([[-0.95,  0.45], [-0.55,  0.10], [-0.55,  0.85], [-0.35, 0.98], [-0.02, 0.62], [-0.32, 0.12]]),
    np.array([[-0.55,  0.10], [-0.02, 0.62], [0.48, 0.78], [0.72, 0.55], [0.48, 0.12], [0.12, -0.02], [-0.32, 0.12]]),
    np.array([[-0.32, 0.12], [0.12, -0.02], [0.35, -0.42], [-0.15, -0.62], [-0.55, -0.55], [-0.55, 0.10]]),
    np.array([[0.12, -0.02], [0.48, 0.12], [0.70, -0.30], [0.35, -0.42]]),
    np.array([[0.48, 0.12], [0.72, 0.55], [1.05, 0.42], [0.85, -0.18], [0.70, -0.30]]),
    np.array([[0.35, -0.42], [0.70, -0.30], [0.85, -0.18], [0.62, -0.92], [0.18, -0.98], [-0.15, -0.62]]),
]

partition_basis_surfaces(partition_polygons, deltas=(0.05, 0.10, 0.20), n=2, N=140)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), squeeze=False)
for idx, (ax, poly) in enumerate(zip(axes[0], partition_polygons[:3]), start=1):
    draw_imp_spline(poly, delta=0.12, n=2, N=240, ax=ax, title=f'Partition cell {idx}')
fig.suptitle('Individual basis functions on partition cells', fontsize=12)
plt.tight_layout(); plt.show()

---
## Notes

* The construction is exact for **convex** polygons.  For non-convex polygons
  the product formula may produce reduced values near reflex vertices;
  increase `delta` or pre-decompose into convex parts.
* See `02_data_from_file.ipynb` for loading polygon vertices from a text file.
